# CDC and Delta MERGE – Retail Sales Capstone


## 1. Configuration and source validation

In [0]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import *

sales_table = "capretailnew.silver.silver_sales_order_lines"
events_table = "capretailnew.silver.cdc_order_status_events_demo"
target_table = "capretailnew.silver.cdc_order_status_current_demo"

if not spark.catalog.tableExists(sales_table):
    raise Exception(f"Required source table not found: {sales_table}")

sales_df = spark.table(sales_table)
required = {"order_id", "order_status", "order_ts"}
missing = required - set(sales_df.columns)
if missing:
    raise Exception(f"Silver sales table is missing columns: {sorted(missing)}")

display(sales_df.select("order_id", "order_status", "order_ts").limit(10))

## 2. Create initial current-state target from Silver

In [0]:
# Select one starting status per order, using latest order_ts.
w = Window.partitionBy("order_id").orderBy(
    F.col("order_ts").desc_nulls_last(),
    F.col("order_status").desc_nulls_last()
)

initial_state = (
    sales_df.filter(F.col("order_id").isNotNull())
    .withColumn("rn", F.row_number().over(w))
    .filter(F.col("rn") == 1)
    .select(
        F.col("order_id").cast("string").alias("order_id"),
        F.col("order_status").cast("string").alias("order_status"),
        F.col("order_ts").cast("timestamp").alias("status_updated_at")
    )
    .withColumn("last_event_id", F.lit("INITIAL_FROM_SILVER"))
)

initial_state.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(target_table)
print("Initial target rows:", spark.table(target_table).count())
display(spark.table(target_table).orderBy("order_id").limit(10))

## 3. Create simulated incoming CDC events

These are **demo events**, not actual source-system streaming records. They use a few real order IDs from the existing Silver table to demonstrate later status updates.

In [0]:
sample_orders = (
    spark.table(target_table).select("order_id").orderBy("order_id").limit(3).collect()
)
if not sample_orders:
    raise Exception("No order IDs found in Silver; CDC demo cannot continue.")

order_ids = [r["order_id"] for r in sample_orders]
rows = []
for i, order_id in enumerate(order_ids):
    rows.append((f"DEMO_EVT_{i+1}_A", order_id, "SHIPPED", "2026-11-05 10:00:00", 1, "UPSERT"))
    rows.append((f"DEMO_EVT_{i+1}_B", order_id, "DELIVERED", "2026-11-05 11:00:00", 2, "UPSERT"))

schema = StructType([
    StructField("event_id", StringType(), False),
    StructField("order_id", StringType(), False),
    StructField("new_status", StringType(), False),
    StructField("event_ts_text", StringType(), False),
    StructField("sequence_no", IntegerType(), False),
    StructField("operation", StringType(), False)
])

events_df = (
    spark.createDataFrame(rows, schema)
    .withColumn("event_ts", F.to_timestamp("event_ts_text"))
    .drop("event_ts_text")
)
events_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(events_table)
print("Demo events:", spark.table(events_table).count())
display(spark.table(events_table).orderBy("order_id", "event_ts"))

## 4. Validate and deterministically deduplicate events

In [0]:
events = spark.table(events_table)

invalid = events.filter(
    F.col("event_id").isNull() |
    F.col("order_id").isNull() |
    F.col("new_status").isNull() |
    F.col("event_ts").isNull() |
    (~F.col("operation").isin("UPSERT"))
)
if invalid.limit(1).count():
    display(invalid)
    raise Exception("Invalid CDC events found.")

duplicate_ids = events.groupBy("event_id").count().filter(F.col("count") > 1)
if duplicate_ids.limit(1).count():
    display(duplicate_ids)
    raise Exception("Duplicate event_id values found.")

latest_w = Window.partitionBy("order_id").orderBy(
    F.col("event_ts").desc(),
    F.col("sequence_no").desc(),
    F.col("event_id").desc()
)
latest_events = (
    events.withColumn("rn", F.row_number().over(latest_w))
    .filter(F.col("rn") == 1)
    .drop("rn")
)
display(latest_events.orderBy("order_id"))

## 5. Apply changes using Delta Lake MERGE

In [0]:
target = DeltaTable.forName(spark, target_table)

(
    target.alias("t")
    .merge(latest_events.alias("s"), "t.order_id = s.order_id")
    .whenMatchedUpdate(
        condition="s.event_ts > t.status_updated_at OR t.status_updated_at IS NULL",
        set={
            "order_status": "s.new_status",
            "status_updated_at": "s.event_ts",
            "last_event_id": "s.event_id"
        }
    )
    .whenNotMatchedInsert(
        values={
            "order_id": "s.order_id",
            "order_status": "s.new_status",
            "status_updated_at": "s.event_ts",
            "last_event_id": "s.event_id"
        }
    )
    .execute()
)

print("CDC MERGE completed.")
display(spark.table(target_table).orderBy("order_id"))

## 6. Validate inserted/updated current state

In [0]:
validation = (
    latest_events.alias("s")
    .join(spark.table(target_table).alias("t"), "order_id", "left")
    .select(
        F.col("order_id"),
        F.col("s.new_status").alias("expected_status"),
        F.col("t.order_status").alias("actual_status"),
        F.col("s.event_id").alias("expected_event_id"),
        F.col("t.last_event_id").alias("actual_event_id")
    )
    .withColumn(
        "validation",
        F.when(
            (F.col("expected_status") == F.col("actual_status")) &
            (F.col("expected_event_id") == F.col("actual_event_id")),
            "PASS"
        ).otherwise("FAIL")
    )
)
display(validation)
if validation.filter(F.col("validation") == "FAIL").count():
    raise Exception("CDC MERGE validation failed.")

dupes = spark.table(target_table).groupBy("order_id").count().filter(F.col("count") > 1)
if dupes.limit(1).count():
    display(dupes)
    raise Exception("Duplicate order_id found in current-state target.")
print("Current-state validation: PASS")

## 7. Replay the same events to demonstrate idempotency

In [0]:
before = spark.table(target_table).select("order_id", "order_status", "status_updated_at", "last_event_id")

target = DeltaTable.forName(spark, target_table)
(
    target.alias("t")
    .merge(latest_events.alias("s"), "t.order_id = s.order_id")
    .whenMatchedUpdate(
        condition="s.event_ts > t.status_updated_at OR t.status_updated_at IS NULL",
        set={
            "order_status": "s.new_status",
            "status_updated_at": "s.event_ts",
            "last_event_id": "s.event_id"
        }
    )
    .whenNotMatchedInsert(
        values={
            "order_id": "s.order_id",
            "order_status": "s.new_status",
            "status_updated_at": "s.event_ts",
            "last_event_id": "s.event_id"
        }
    )
    .execute()
)

after = spark.table(target_table).select("order_id", "order_status", "status_updated_at", "last_event_id")
changed = (
    before.alias("b").join(after.alias("a"), "order_id", "full")
    .filter(
        ~F.col("b.order_status").eqNullSafe(F.col("a.order_status")) |
        ~F.col("b.status_updated_at").eqNullSafe(F.col("a.status_updated_at")) |
        ~F.col("b.last_event_id").eqNullSafe(F.col("a.last_event_id"))
    ).count()
)
print("Rows changed on replay:", changed)
if changed != 0:
    raise Exception("Replay validation failed.")
print("Replay/idempotency validation: PASS")

## 8. Inspect Delta history

In [0]:
print("Current-state target history:")
display(DeltaTable.forName(spark, target_table).history(10))

print("CDC event source history:")
display(DeltaTable.forName(spark, events_table).history(10))

## 9. Completion and project placement

Expected:
- Existing Silver tables remain unchanged.
- A separate simulated CDC events Delta table is created.
- A separate current-state Delta table is maintained using `MERGE`.
- Latest event per order is selected deterministically.
- Target key uniqueness, MERGE results, replay behaviour, and Delta history are checked.

**Project flow:** Bronze → existing Silver notebooks → **this CDC MERGE notebook** → Gold notebooks.

**Production extension:** If the trainer later requires a continuously running stream, replace the demo event-generation section with the real CDC stream and invoke the same MERGE logic inside `foreachBatch`. This notebook demonstrates batch CDC processing; it does not claim that the project is currently running a stream.